In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import glob
import os
import polars as pl
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import pickle
import ctypes
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

In [ ]:
feature_file_path = '/kaggle/input/birdclef-2025-features-labels-v1/extracted_train_feat_five_sec_v5_2025'
label_file_path = '/kaggle/input/birdclef-2025-features-labels-v1/labels_five_sec_v5_2025'

missing_classes_feature_file_path = '/kaggle/input/missing-classes-features-labels/late_inclusion_features_2025'
missing_classes_label_file_path = '/kaggle/input/missing-classes-features-labels/late_inclusion_labels_2025'

with open(feature_file_path, "rb") as file:
    pickled_extracted_features_five_sec = pickle.load(file)
    
with open(label_file_path, "rb") as file:
    labels_five_sec = pickle.load(file)

with open(missing_classes_feature_file_path, "rb") as file:
    pickled_missing_classes_features_five_sec = pickle.load(file)
    
with open(missing_classes_label_file_path, "rb") as file:
    labels_missing_classes_five_sec = pickle.load(file)

In [ ]:
x_five_sec = np.vstack([pickled_extracted_features_five_sec, pickled_missing_classes_features_five_sec])
y_five_sec = np.vstack([labels_five_sec, labels_missing_classes_five_sec])

print("Combined features shape:", x_five_sec.shape)
print("Combined labels shape:", y_five_sec.shape)

In [ ]:
def create_model(input_shape, num_classes):
    model = models.Sequential([
        # Input layer
        layers.Input(shape=input_shape),
        
        # Dense layers for classification
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

In [ ]:
def prepare_dataset(X, y, batch_size=32):
    """
    Convert numpy arrays to tf.data.Dataset with batching and prefetching
    """
    # Normalize features
    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0) + 1e-8
    X_normalized = (X - X_mean) / X_std
    
    # Convert labels to categorical
    """label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    y_categorical = tf.keras.utils.to_categorical(y_encoded)"""
    
    # Create tf.data.Dataset
    #dataset = tf.data.Dataset.from_tensor_slices((X_normalized, y_categorical))
    dataset = tf.data.Dataset.from_tensor_slices((X_normalized, y))
    dataset = dataset.cache()  # Cache the data in memory
    dataset = dataset.shuffle(buffer_size=len(X))  # Shuffle the entire dataset
    dataset = dataset.batch(batch_size)  # Batch the data
    dataset = dataset.prefetch(tf.data.AUTOTUNE)  # Prefetch next batch
    
    return dataset, label_encoder

In [ ]:
def train_model_tpu(X, y, batch_size=32, epochs=50):
    """
    Train the model using TPU acceleration
    """
    # Initialize TPU
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        print('Running on TPU ', tpu.cluster_spec().as_dict()['worker'])
        
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.TPUStrategy(tpu)
        print("Number of accelerators: ", strategy.num_replicas_in_sync)
    except:
        print('No TPU detected. Using GPU/CPU strategy')
        strategy = tf.distribute.get_strategy()

    # Split the data
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Prepare datasets
    train_dataset = prepare_dataset(X_train, y_train, batch_size)
    val_dataset = prepare_dataset(X_val, y_val, batch_size)
    
    # Get input shape and number of classes
    input_shape = (X.shape[1],)  # Will be (40,)
    num_classes = len(label_encoder.classes_)
    
    # Create and compile model using TPU strategy
    with strategy.scope():
        model = create_model(input_shape, num_classes)
        model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
    
    # Callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=3,
            min_lr=1e-6
        )
    ]
    
    # Train the model
    history = model.fit(
        train_dataset,
        epochs=epochs,
        validation_data=val_dataset,
        callbacks=callbacks
    )
    
    return model, history, label_encoder